In [0]:
"""
05_packaging_kpis.py

Manufacturing Packaging KPIs

Source:
    fact_packaging

Target:
    packaging_kpis

Author:
Sumanth Vempalle

Version:
2.0.0
"""

import dlt

from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    sum,
    when,
)

# ============================================================
# Packaging KPIs
# ============================================================

@dlt.table(
    name="packaging_kpis",
    comment="Manufacturing packaging KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def packaging_kpis():

    packaging = dlt.read("fact_packaging")

    return (

        packaging

        .groupBy(

            "plant_code",
            "product_code",
            "product_name",
            "family",
            "package_type",

        )

        .agg(

            count("*").alias(
                "packages_completed"
            ),

            avg(
                "package_weight_kg"
            ).alias(
                "average_package_weight_kg"
            ),

            avg(
                "package_length_mm"
            ).alias(
                "average_package_length_mm"
            ),

            avg(
                "package_width_mm"
            ).alias(
                "average_package_width_mm"
            ),

            avg(
                "package_height_mm"
            ).alias(
                "average_package_height_mm"
            ),

            sum(

                when(

                    col("packaging_status") == "READY_FOR_SHIPMENT",

                    1

                ).otherwise(0)

            ).alias(
                "ready_for_shipment"
            ),

            (
                count("*")

                -

                sum(

                    when(

                        col("packaging_status") == "READY_FOR_SHIPMENT",

                        1

                    ).otherwise(0)

                )

            ).alias(
                "not_ready_for_shipment"
            ),

        )

        .withColumn(

            "shipment_readiness_rate",

            when(

                col("packages_completed") > 0,

                col("ready_for_shipment")
                * 100.0
                / col("packages_completed")

            ).otherwise(0.0)

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )